In [41]:
# ==========================================
# Merge PROF + accounting noise
# Build uncertainty-adjusted quality signals
# Formation years: 2010–2025
# ==========================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(".").resolve()

PROF_DIR = BASE_DIR / "prof_components_extracted"
SIGMA_FILE = BASE_DIR / "acc_noise_ols_no_lead" / "firm_year_residuals_and_sigma_no_lead_avgat.csv"

OUT_DIR = BASE_DIR / "quality_signals_eb"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# User choices
# -----------------------------
FORMATION_YEAR_MIN = 2010
FORMATION_YEAR_MAX = 2025

# Main sigma column from the OLS step
SIGMA_COL = "sigma_acc"          # alternative: "sigma_acc_abs"

# How strongly sigma_acc should map into PROF observation variance.
# Interpretation:
# median(obs_var) = NOISE_SHARE_OF_PROF_VAR * var(PROF) within each year
NOISE_SHARE_OF_PROF_VAR = 0.15

# Winsorization settings
WINSORIZE_PROF = True
WINSORIZE_SIGMA = True
WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99

MIN_FIRMS_PER_YEAR = 20


In [42]:

# -----------------------------
# Helpers
# -----------------------------
def winsorize_series(s: pd.Series, lower=0.01, upper=0.99) -> pd.Series:
    if s.notna().sum() == 0:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lower=lo, upper=hi)

# -----------------------------
# 1. Load PROF panel
# -----------------------------
prof_files = sorted(PROF_DIR.glob("*.csv"))
print(f"Found {len(prof_files)} prof files in {PROF_DIR}")

prof_dfs = []
for fp in prof_files:
    try:
        df = pd.read_csv(fp)
        prof_dfs.append(df)
    except Exception as e:
        print(f"Skipping {fp.name}: {e}")

prof_panel = pd.concat(prof_dfs, ignore_index=True)
print(f"Combined PROF panel shape: {prof_panel.shape}")

# Keep relevant columns
keep_cols = [
    "Year", "Ticker", "CompanyName", "Industry", "Sector",
    "PROF", "REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT", "BE", "MIB"
]
keep_cols = [c for c in keep_cols if c in prof_panel.columns]
prof_panel = prof_panel[keep_cols].copy()

# Clean
prof_panel["Year"] = pd.to_numeric(prof_panel["Year"], errors="coerce")
for c in [c for c in prof_panel.columns if c not in ["Ticker", "CompanyName", "Industry", "Sector"]]:
    if c != "Year":
        prof_panel[c] = pd.to_numeric(prof_panel[c], errors="coerce")

prof_panel = prof_panel.sort_values(["Ticker", "Year"]).reset_index(drop=True)


Found 635 prof files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/prof_components_extracted
Combined PROF panel shape: (10387, 16)


In [43]:

# -----------------------------
# 2. Load accounting-noise panel
# -----------------------------
sigma_df = pd.read_csv(SIGMA_FILE)
print(f"Loaded sigma file shape: {sigma_df.shape}")

sigma_df["Year"] = pd.to_numeric(sigma_df["Year"], errors="coerce")

sigma_keep = ["Ticker", "Year", "dd_resid", "sigma_acc", "sigma_acc_abs"]
sigma_keep = [c for c in sigma_keep if c in sigma_df.columns]
sigma_df = sigma_df[sigma_keep].copy()


Loaded sigma file shape: (9275, 14)


In [44]:

# -----------------------------
# 3. Merge PROF + sigma
# -----------------------------
signal_df = prof_panel.merge(
    sigma_df,
    on=["Ticker", "Year"],
    how="left",
    validate="1:1"
)

signal_df["FormationYear"] = signal_df["Year"] + 1

# Keep only desired formation years
signal_df = signal_df[
    (signal_df["FormationYear"] >= FORMATION_YEAR_MIN) &
    (signal_df["FormationYear"] <= FORMATION_YEAR_MAX)
].copy()

print(f"Merged signal panel shape after formation-year filter: {signal_df.shape}")


Merged signal panel shape after formation-year filter: (8422, 17)


In [45]:

# -----------------------------
# 4. Basic sample restrictions
# -----------------------------
signal_df = signal_df.replace([np.inf, -np.inf], np.nan)

# Need PROF and sigma for uncertainty-adjusted methods
signal_df = signal_df.dropna(subset=["PROF", SIGMA_COL]).copy()

print(f"Shape after requiring PROF and {SIGMA_COL}: {signal_df.shape}")

# Optional winsorization within formation year
if WINSORIZE_PROF:
    signal_df["PROF_w"] = (
        signal_df.groupby("FormationYear", group_keys=False)["PROF"]
        .apply(lambda s: winsorize_series(s, WINSOR_LOWER, WINSOR_UPPER))
    )
else:
    signal_df["PROF_w"] = signal_df["PROF"]

if WINSORIZE_SIGMA:
    signal_df["sigma_raw"] = (
        signal_df.groupby("FormationYear", group_keys=False)[SIGMA_COL]
        .apply(lambda s: winsorize_series(s, WINSOR_LOWER, WINSOR_UPPER))
    )
else:
    signal_df["sigma_raw"] = signal_df[SIGMA_COL]

# Ensure positive sigma
signal_df = signal_df[signal_df["sigma_raw"] > 0].copy()

print(f"Shape after sigma positivity filter: {signal_df.shape}")


Shape after requiring PROF and sigma_acc: (7292, 17)
Shape after sigma positivity filter: (7292, 19)


In [46]:

# -----------------------------
# 5. Empirical Bayes adjustment by formation year
# -----------------------------
year_results = []
frames = []

for fy, sub in signal_df.groupby("FormationYear"):
    sub = sub.copy()

    if len(sub) < MIN_FIRMS_PER_YEAR:
        print(f"Skipping FormationYear={fy}: only {len(sub)} firms")
        continue

    # Observed signal
    theta_obs = sub["PROF_w"].astype(float)

    # Cross-sectional prior for latent quality
    mu_t = theta_obs.mean()
    var_obs = theta_obs.var(ddof=1)

    if not np.isfinite(var_obs) or var_obs <= 0:
        print(f"Skipping FormationYear={fy}: invalid observed PROF variance")
        continue

    # Map sigma_acc into observation variance for PROF
    # Step 1: use sigma relative to median sigma in the same year
    sigma_rel = sub["sigma_raw"] / sub["sigma_raw"].median()

    # Step 2: set observation variance so that the median firm has
    # obs_var = NOISE_SHARE_OF_PROF_VAR * var(PROF)
    obs_var_base = NOISE_SHARE_OF_PROF_VAR * var_obs
    obs_var_i = obs_var_base * (sigma_rel ** 2)

    # Latent cross-sectional variance:
    # observed variance = latent variance + average observation variance
    avg_obs_var = obs_var_i.mean()
    tau2_t = max(var_obs - avg_obs_var, 1e-8)

    # Posterior mean shrinkage weight
    lambda_i = tau2_t / (tau2_t + obs_var_i)

    theta_post_mean = lambda_i * theta_obs + (1 - lambda_i) * mu_t

    # Posterior variance of latent theta*
    post_var_i = (tau2_t * obs_var_i) / (tau2_t + obs_var_i)
    post_sd_i = np.sqrt(post_var_i)

    # Define Q5 threshold using observed PROF distribution in that formation year
    q5_cutoff = theta_obs.quantile(0.80)

    # Posterior probability latent theta* is in Q5
    z = (q5_cutoff - theta_post_mean) / post_sd_i
    p_q5 = 1 - norm.cdf(z)

    # Store
    sub["theta_obs"] = theta_obs
    sub["mu_t"] = mu_t
    sub["var_obs_t"] = var_obs
    sub["obs_var_i"] = obs_var_i
    sub["tau2_t"] = tau2_t
    sub["lambda_i"] = lambda_i
    sub["theta_post_mean"] = theta_post_mean
    sub["theta_post_sd"] = post_sd_i
    sub["q5_cutoff_obs"] = q5_cutoff
    sub["p_q5"] = p_q5

    frames.append(sub)

    year_results.append({
        "FormationYear": fy,
        "n_firms": len(sub),
        "mu_t": mu_t,
        "var_obs_t": var_obs,
        "avg_obs_var_i": avg_obs_var,
        "tau2_t": tau2_t,
        "q5_cutoff_obs": q5_cutoff,
    })

signals_eb_df = pd.concat(frames, ignore_index=True).sort_values(["FormationYear", "Ticker"]).reset_index(drop=True)
signals_year_df = pd.DataFrame(year_results).sort_values("FormationYear").reset_index(drop=True)

print("\nYear-level EB summary:")
display(signals_year_df)

print("\nSignal preview:")
display(
    signals_eb_df[
        [
            "Ticker", "Year", "FormationYear",
            "PROF", "theta_obs", "sigma_raw",
            "lambda_i", "theta_post_mean", "p_q5"
        ]
    ].head(20)
)



Year-level EB summary:


,FormationYear,n_firms,mu_t,var_obs_t,avg_obs_var_i,tau2_t,q5_cutoff_obs
0,2010,307,0.123974,0.265330,0.169338,0.095992,0.360747
1,2011,324,0.147215,0.191823,0.116409,0.075415,0.372639
2,2012,339,0.109394,0.229997,0.138248,0.091749,0.345885
3,2013,356,0.102300,0.159441,0.076712,0.082729,0.323707
4,2014,375,0.062333,0.220018,0.129303,0.090715,0.272913
5,2015,389,0.105836,0.170773,0.107144,0.063629,0.301449
6,2016,420,0.076527,0.251319,0.169239,0.082080,0.285005
7,2017,432,0.112347,0.221023,0.127919,0.093104,0.314248
8,2018,463,0.067630,0.203738,0.125576,0.078162,0.285106
9,2019,492,0.064140,0.335657,0.295591,0.040067,0.299015



Signal preview:


,Ticker,Year,FormationYear,PROF,theta_obs,sigma_raw,lambda_i,theta_post_mean,p_q5
0,AAB.CO,2009,2010,-0.528651,-0.528651,0.085812,0.609567,-0.273844,5.227990e-04
1,AAK.ST,2009,2010,0.452191,0.452191,0.069041,0.706907,0.355993,4.886940e-01
2,ABB.ST,2009,2010,-1.330641,-1.330641,0.065651,0.727329,-0.934010,5.551115e-16
3,ACG1V.HE,2009,2010,-1.242334,-1.242334,0.070052,0.700851,-0.833604,9.069412e-13
4,ACTI.ST,2009,2010,-0.117801,-0.117801,0.111394,0.480925,0.007698,5.686783e-02
5,ADDTb.ST,2009,2010,0.401306,0.401306,0.051922,0.810049,0.348627,4.642381e-01
6,AFAGR.HE,2009,2010,-0.100644,-0.100644,0.528236,0.039572,0.115086,2.092367e-01
7,AFGA.OL,2009,2010,0.334765,0.334765,0.077218,0.658484,0.262777,2.942213e-01
8,AFK.OL,2009,2010,0.056797,0.056797,0.144527,0.355002,0.100126,1.474572e-01
9,AFRY.ST,2009,2010,0.227196,0.227196,0.036853,0.894346,0.216290,7.572437e-02


In [47]:

# -----------------------------
# 6. Create portfolio-sorting versions
# -----------------------------
# Method 1: raw theta
signals_eb_df["sort_signal_m1"] = signals_eb_df["theta_obs"]

# Method 2: posterior mean
signals_eb_df["sort_signal_m2"] = signals_eb_df["theta_post_mean"]

# Method 3: posterior probability of Q5
signals_eb_df["sort_signal_m3"] = signals_eb_df["p_q5"]

# Optional: within-year ranks
for method_col, rank_col in [
    ("sort_signal_m1", "rank_m1"),
    ("sort_signal_m2", "rank_m2"),
    ("sort_signal_m3", "rank_m3"),
]:
    signals_eb_df[rank_col] = (
        signals_eb_df.groupby("FormationYear")[method_col]
        .rank(method="average", pct=True)
    )

# Optional: assign quintiles within year
def assign_quintiles(s: pd.Series) -> pd.Series:
    # rank first to avoid qcut issues with ties
    pct = s.rank(method="average", pct=True)
    q = pd.cut(
        pct,
        bins=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        labels=[1, 2, 3, 4, 5],
        include_lowest=True
    )
    return q.astype("Int64")

signals_eb_df["q_m1"] = signals_eb_df.groupby("FormationYear")["sort_signal_m1"].transform(assign_quintiles)
signals_eb_df["q_m2"] = signals_eb_df.groupby("FormationYear")["sort_signal_m2"].transform(assign_quintiles)
signals_eb_df["q_m3"] = signals_eb_df.groupby("FormationYear")["sort_signal_m3"].transform(assign_quintiles)

print("\nQuintile counts preview:")
display(
    signals_eb_df.groupby("FormationYear")[["q_m1", "q_m2", "q_m3"]]
    .agg(lambda x: x.value_counts().sort_index().to_dict())
    .head()
)



Quintile counts preview:


,q_m1,q_m2,q_m3
FormationYear,,,
2010,"{1: 61, 2: 61, 3: 62, 4: 61, 5: 62}","{1: 61, 2: 61, 3: 62, 4: 61, 5: 62}","{1: 61, 2: 61, 3: 62, 4: 61, 5: 62}"
2011,"{1: 64, 2: 65, 3: 65, 4: 65, 5: 65}","{1: 64, 2: 65, 3: 65, 4: 65, 5: 65}","{1: 64, 2: 65, 3: 65, 4: 65, 5: 65}"
2012,"{1: 67, 2: 68, 3: 68, 4: 68, 5: 68}","{1: 67, 2: 68, 3: 68, 4: 68, 5: 68}","{1: 67, 2: 68, 3: 68, 4: 68, 5: 68}"
2013,"{1: 71, 2: 71, 3: 71, 4: 71, 5: 72}","{1: 71, 2: 71, 3: 71, 4: 71, 5: 72}","{1: 71, 2: 71, 3: 71, 4: 71, 5: 72}"
2014,"{1: 75, 2: 75, 3: 75, 4: 75, 5: 75}","{1: 75, 2: 75, 3: 75, 4: 75, 5: 75}","{1: 75, 2: 75, 3: 75, 4: 75, 5: 75}"


In [48]:

# -----------------------------
# 7. Save outputs
# -----------------------------
signals_eb_df.to_csv(OUT_DIR / "firm_year_quality_signals_eb.csv", index=False)
signals_year_df.to_csv(OUT_DIR / "year_level_eb_summary.csv", index=False)

print(f"\nSaved outputs to: {OUT_DIR}")


Saved outputs to: /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/quality_signals_eb
